In [1]:
import os
import time
import json
from typing import TypedDict, List
from dotenv import load_dotenv

In [2]:
%pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.


c:\Users\LENOVO\Desktop\rag_agentic\.venv\Scripts\python.exe: No module named pip


In [3]:
# ─── Récupérer les Clés API ──────────────────────────────────────────
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "VOTRE_CLE_GROQ_ICI")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "VOTRE_CLE_TAVILY_ICI")
# ─── Utiliser le modele Groq
GROQ_MODEL = "openai/gpt-oss-20b"
GROQ_OK = GROQ_API_KEY.startswith("gsk_") and len(GROQ_API_KEY) > 30
TAVILY_OK = TAVILY_API_KEY.startswith("tvly-") and len(TAVILY_API_KEY) > 20
print(f"Groq : {'✅ ok' if GROQ_OK else '❌ clé manquante'}")
print(f"Tavily : {'✅ ok' if TAVILY_OK else '❌ clé manquante (web search désactivée)'}")
print(f"Modèle : {GROQ_MODEL}")

Groq : ✅ ok
Tavily : ✅ ok
Modèle : openai/gpt-oss-20b


In [2]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19904\25192327.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [5]:
%pip install python-chroma

Note: you may need to restart the kernel to use updated packages.


c:\Users\LENOVO\Desktop\rag_agentic\.venv\Scripts\python.exe: No module named pip


In [11]:

from pathlib import Path

# 1. Définir le contenu des documents
CORPUS = {
    "python.md": """
# Python
Python est un langage de programmation polyvalent.
Il est utilisé pour le développement web, l'analyse de données
et l'intelligence artificielle.
""",
    "machine_learning.md": """
# Machine Learning
Le machine learning est une branche de l'intelligence artificielle.
Il permet aux ordinateurs d'apprendre à partir de données.
""",
    "rag.md": """
# RAG
RAG signifie Retrieval-Augmented Generation.
Cette technique récupère des documents pertinents
pour améliorer les réponses d'un modèle de langage.
""",
    "langchain.md": """
# LangChain
LangChain est un framework permettant de construire
des applications basées sur les modèles de langage,
notamment des systèmes RAG.
"""
}

# 2. Créer le dossier documents/
docs_dir = Path("documents")
docs_dir.mkdir(exist_ok=True)

# 3. Créer les quatre fichiers Markdown
for name, text in CORPUS.items():
    (docs_dir / name).write_text(text.strip(), encoding="utf-8")

print("Les 4 fichiers ont été créés dans le dossier documents.")

# 4. Vérifier les fichiers
for file in sorted(docs_dir.glob("*.md")):
    print(f"- {file.name}")


Les 4 fichiers ont été créés dans le dossier documents.
- langchain.md
- machine_learning.md
- python.md
- rag.md


In [19]:
print("docs :", "docs" in globals())
print("chunks :", "chunks" in globals())
print("embeddings :", "embeddings" in globals())
print("vectorstore :", "vectorstore" in globals())

docs : True
chunks : True
embeddings : True
vectorstore : False


In [27]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1404.06it/s]


In [31]:
from langchain_core.tools import Tool
# ─── Outil 1 : retriever local ───────────────────────────────────────
# On wrappe le retriever Chroma en outil LangChain "à la main" pour rester
# indépendant de l'emplacement (qui bouge entre versions) de
# `create_retriever_tool`.
def _rag_search(query: str) -> str:
    docs = retriever.invoke(query)
    if not docs:return "[Aucun document pertinent trouvé dans le corpus local]"
    return "\n\n---\n\n".join(f"[source: {d.metadata.get('source', '?')}]\n{d.page_content}"
    for d in docs
    )
rag_tool = Tool(
name="rag_search",
description=(
    "Cherche des informations dans le corpus IA local (Transformer, RAG, "
    "fine-tuning, vector databases). À utiliser pour les questions techniques "
    "sur ces sujets quand l'information est susceptible d'être dans la base."
    ),
func=_rag_search,
)
# Test rapide
print(rag_tool.invoke("How does HNSW work?")[:400])

NameError: name 'retriever' is not defined

In [32]:
# ─── 1. Mini corpus IA ───────────────────────────────
# Pour rester compact, on inline 4 textes (au lieu de lire 4 fichiers .md).
CORPUS = {
"transformers.md": '''# Transformer Architecture
## Self-Attention
The Transformer architecture (Vaswani et al., 2017) replaced RNNs with self-attention.
It computes Query (Q), Key (K), Value (V) matrices and uses softmax(QK^T / sqrt(d_k)) * V.
Multi-head attention runs multiple attention heads in parallel.
## Variants
Encoder-only models (BERT), decoder-only models (GPT), encoder-decoder (T5, BART).
Modern positional encodings: RoPE (Rotary), ALiBi.''',
"rag_systems.md": '''# Retrieval-Augmented Generation (RAG)
## Definition
RAG combines an information retrieval layer with an LLM. Responses are generated from
retrieved evidence rather than parametric memory only.
## Hybrid Retrieval
Hybrid retrieval combines sparse (BM25) and dense (embeddings) methods, fused with
Reciprocal Rank Fusion (RRF).
## Chunking
Common strategies: fixed-size, recursive, semantic, structure-aware (Markdown headers).
Optimal chunk_size depends on the use case: 256 tokens for precision, 1024 for context.''',
"vector_databases.md": '''# Vector Databases and Indexing
## HNSW (Hierarchical Navigable Small World)
HNSW builds a multi-layer graph where each layer is a small-world network.
Search starts at the top sparse layer and refines through lower dense layers.
Achieves 95%+ recall with millisecond latency. Key params: M, ef_construction, ef_search.
## IVF (Inverted File Index)
IVF clusters vectors with k-means; search only in nearest clusters (nprobe).
Combined with PQ (product quantization) for memory savings on huge corpora.''',
"finetuning.md": '''# LLM Fine-Tuning
## LoRA (Low-Rank Adaptation)
LoRA freezes the pre-trained model and injects trainable rank decomposition matrices
into each Transformer layer. Update is W + BA where B,A are small.
## QLoRA
QLoRA combines LoRA with 4-bit quantization. Enables fine-tuning 65B models on a single 48GB GPU.
## RLHF and DPO
RLHF: reward model + PPO. DPO: simpler, no separate reward model, classification loss on preference
pairs.''',
}
from langchain_core.documents import Document
docs = [Document(page_content=text, metadata={"source": name})for name, text in CORPUS.items()]
# ─── 2. Splitting ─────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
     encoding_name="cl100k_base", chunk_size=300, chunk_overlap=30,
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} documents → {len(chunks)} chunks")
# ─── 3. Embeddings + Chroma ───────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="tp4_corpus")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Retriever prêt — collection 'tp4_corpus' avec {vectorstore._collection.count()} chunks.")

4 documents → 4 chunks


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 404.79it/s]


Retriever prêt — collection 'tp4_corpus' avec 4 chunks.


In [33]:
from langchain_core.tools import Tool
# ─── Outil 1 : retriever local ────────────────────────────────────
# On wrappe le retriever Chroma en outil LangChain "à la main" pour rester
# indépendant de l'emplacement (qui bouge entre versions) de
# `create_retriever_tool`.
def _rag_search(query: str) -> str:
    docs = retriever.invoke(query)
    if not docs:return "[Aucun document pertinent trouvé dans le corpus local]"
    return "\n\n---\n\n".join(
    f"[source: {d.metadata.get('source', '?')}]\n{d.page_content}"
    for d in docs
    )
rag_tool = Tool(
name="rag_search",
description=(
"Cherche des informations dans le corpus IA local (Transformer, RAG, "
"fine-tuning, vector databases). À utiliser pour les questions techniques "
"sur ces sujets quand l'information est susceptible d'être dans la base."
),
func=_rag_search,
)
# Test rapide
print(rag_tool.invoke("How does HNSW work?")[:400])


[source: vector_databases.md]
# Vector Databases and Indexing
## HNSW (Hierarchical Navigable Small World)
HNSW builds a multi-layer graph where each layer is a small-world network.
Search starts at the top sparse layer and refines through lower dense layers.
Achieves 95%+ recall with millisecond latency. Key params: M, ef_construction, ef_search.
## IVF (Inverted File Index)
IVF clusters vectors 


In [45]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun
import os

In [46]:
TAVILY_OK = False
 
try:
    from langchain_community.tools.tavily_search import TavilySearchResults
 
    TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "")
 
    if TAVILY_API_KEY:
        TAVILY_OK = True
        print("✅ Tavily disponible")
    else:
        print("⚠️ Clé API Tavily manquante")
 
except ImportError:
    print("⚠️ Module Tavily non installé")
 
if TAVILY_OK:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
 
    tavily_tool = TavilySearchResults(
        max_results=3,
        include_answer=True
    )
 
    print("✅ Outil Tavily prêt")
 
else:
    tavily_tool = None
    print("⚠️ Recherche web désactivée")
 

✅ Tavily disponible
✅ Outil Tavily prêt


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1964\2534306620.py:20: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


In [47]:
import os
# ─── Outil 2 : recherche web (Tavily) ─────────────────────────────────────
if TAVILY_OK:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY # pour le client interne
    tavily_tool = TavilySearchResults(
    max_results=3,
    name="tavily_web_search",
    description=(
    "Recherche sur le web public. À utiliser pour les questions récentes, "
        "hors du corpus local, ou nécessitant des informations à jour."
),
)
else:
# Stub si pas de clé Tavily (pour que le notebook tourne quand même)
    tavily_tool = Tool.from_function(
func=lambda q: "[Tavily désactivé — pas de clé API]",
name="tavily_web_search",
description="Recherche web (désactivée).",
)
# ─── Outil 3 : arXiv (papiers scientifiques) ──────────────────────────────
arxiv_tool = ArxivQueryRun()
arxiv_tool.name = "arxiv_search"
arxiv_tool.description = (
"Cherche des papiers scientifiques sur arXiv. À utiliser pour les questions "
"concernant des publications académiques précises (auteur, titre, papier précis)."
)
tools = [rag_tool, tavily_tool, arxiv_tool]
for t in tools:
    print(f" • {t.name:<25} — {t.description[:80]}...")

 • rag_search                — Cherche des informations dans le corpus IA local (Transformer, RAG, fine-tuning,...
 • tavily_web_search         — Recherche sur le web public. À utiliser pour les questions récentes, hors du cor...
 • arxiv_search              — Cherche des papiers scientifiques sur arXiv. À utiliser pour les questions conce...


In [48]:
# System prompt de l'agent — versionné comme du code.
# Avec le `create_react_agent` de LangGraph (utilisé dans la cellule suivante),
# on passe une simple chaîne. La boucle Thought → Action → Observation est
# gérée nativement par les `tool_calls` du LLM (plus besoin de format texte
# fragile à parser comme dans l'ancien ReAct LangChain).
SYSTEM_PROMPT = """Tu es un AI Research Assistant spécialisé en intelligence artificielle.
Tu disposes de 3 outils :
- `rag_search` : corpus IA local (Transformer, RAG, HNSW, LoRA, vector DBs).
- `tavily_web_search` : recherche web généraliste (infos récentes 2024-2026).
- `arxiv_search` : papiers scientifiques arXiv (publications académiques précises).
RÈGLES :
- Utilise `rag_search` en priorité pour les concepts fondamentaux IA déjà bien établis
(Transformer, RAG, HNSW, LoRA, embeddings...).
- Utilise `tavily_web_search` pour les informations récentes (2024-2026) ou hors corpus local.
- Utilise `arxiv_search` quand la question concerne un papier précis ou des publications
académiques.
- Refuse poliment les questions hors-domaine (cuisine, météo, etc.) en français,
SANS appeler aucun outil.
- Cite tes sources dans ta réponse finale.
- Réponds en français.
"""
print(SYSTEM_PROMPT[:300], "...")

Tu es un AI Research Assistant spécialisé en intelligence artificielle.
Tu disposes de 3 outils :
- `rag_search` : corpus IA local (Transformer, RAG, HNSW, LoRA, vector DBs).
- `tavily_web_search` : recherche web généraliste (infos récentes 2024-2026).
- `arxiv_search` : papiers scientifiques arXiv  ...


In [49]:
import os
from langchain_groq import ChatGroq
 
# Configuration Groq
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL = "openai/gpt-oss-20b"
 
# Vérifier la clé API
if not GROQ_API_KEY:
    raise ValueError(
        "Clé GROQ_API_KEY manquante. Configure ta clé API Groq."
    )
 
# Initialiser le modèle
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0
)
 
print("✅ Modèle Groq initialisé :", GROQ_MODEL)

✅ Modèle Groq initialisé : openai/gpt-oss-20b


In [50]:
golden_set = [
    {"id": "Q1",
    "question": "Comment fonctionne l'algorithme HNSW pour la recherche approximative de plus proches voisins ?",
    "expected_tools": ["rag_search"]},
{"id": "Q2",
"question": "Quels nouveaux modèles d'embedding open-source sont sortis en 2025 ?",
"expected_tools": ["tavily_web_search"]},
{"id": "Q3","question": "Que dit le papier original LoRA (2021) sur l'efficacité paramétrique du fine-tuning ?",
 
"expected_tools": ["arxiv_search"]},
{"id": "Q4","question": "Compare HNSW à des alternatives utilisées en production en 2024-2025.",
"expected_tools": ["rag_search", "tavily_web_search"]},
{"id": "Q5",
 "question": "Quelle est la recette préférée de l''auteur du papier 'Attention is all you need'?",
"expected_tools": []}, # On attend un refus poli
]
print(f"{len(golden_set)} questions du golden set prêtes.")

5 questions du golden set prêtes.
